# FraudShield S3 Structure Checker

This notebook validates that your S3 bucket has the correct folder structure for the FraudShield lecture series. If files were uploaded to the wrong location (e.g., bucket root instead of the `fraudshield/` prefix), this notebook will detect and move them.

**Run this before starting any Week 2+ notebook.**

### Expected Structure

```
s3://<your-default-bucket>/
  fraudshield/
    data/
      train/
        train.csv
      validation/
        validation.csv
    output/
      <training-job>/
        output/
          model.tar.gz
    feature-store/
    autopilot/
```

In [ ]:
import boto3
import sagemaker
from sagemaker.session import Session

session = Session()
bucket = session.default_bucket()
prefix = "fraudshield"
s3 = boto3.client("s3")

print(f"Bucket: s3://{bucket}")
print(f"Prefix: {prefix}/")
print("="*60)

In [ ]:
EXPECTED_FILES = {
    "train.csv": {
        "correct_key": f"{prefix}/data/train/train.csv",
        "description": "Training dataset",
        "source": "W1 Friday notebook -- Step: Upload Data to S3",
        "fallback": "W2 Monday notebook -- Fallback cell regenerates data and retrains",
    },
    "validation.csv": {
        "correct_key": f"{prefix}/data/validation/validation.csv",
        "description": "Validation dataset",
        "source": "W1 Friday notebook -- Step: Upload Data to S3",
        "fallback": "W2 Monday notebook -- Fallback cell regenerates data and retrains",
    },
    "model.tar.gz": {
        "correct_prefix": f"{prefix}/output/",
        "description": "Random Forest model artifact",
        "source": "W1 Friday notebook -- Step: Launch Training Job",
        "fallback": "W2 Monday notebook -- Fallback cell retrains the model",
    },
}

print("Expected files and their correct locations:")
print("-" * 60)
for name, info in EXPECTED_FILES.items():
    loc = info.get("correct_key", info.get("correct_prefix") + "<job-name>/output/" + name)
    print(f"  {name:25s} -> {loc}")

In [ ]:
def list_all_keys(bucket, prefix=""):
    """Return all S3 keys under the given prefix."""
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def find_misplaced(bucket, filename, correct_prefix):
    """Find copies of `filename` that are outside the correct prefix."""
    all_keys = list_all_keys(bucket)
    misplaced = []
    for key in all_keys:
        if key.endswith(filename) and not key.startswith(correct_prefix):
            misplaced.append(key)
    return misplaced


def copy_s3_object(bucket, source_key, dest_key):
    """Copy an object within the same bucket."""
    s3.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": source_key},
        Key=dest_key,
    )


print("Helper functions loaded.")

---
## Check 1: Scan the bucket and report current state

This cell scans the entire bucket and reports what it finds: files in the correct location, files in the wrong location, and files that are missing entirely.

In [ ]:
all_keys = list_all_keys(bucket)

print(f"Total objects in bucket: {len(all_keys)}")
print(f"Objects under '{prefix}/': {len([k for k in all_keys if k.startswith(prefix + '/')])}")
print(f"Objects at root (outside '{prefix}/'): {len([k for k in all_keys if not k.startswith(prefix + '/')])}")
print("=" * 60)

STATUS_OK = "OK"
STATUS_MISPLACED = "MISPLACED"
STATUS_MISSING = "MISSING"

results = {}

for filename, info in EXPECTED_FILES.items():
    correct_key = info.get("correct_key")
    correct_pfx = info.get("correct_prefix", f"{prefix}/")

    in_correct_spot = False
    if correct_key:
        in_correct_spot = correct_key in all_keys
    else:
        in_correct_spot = any(
            k.startswith(correct_pfx) and k.endswith(filename)
            for k in all_keys
        )

    misplaced = find_misplaced(bucket, filename, correct_pfx)

    if in_correct_spot:
        results[filename] = {"status": STATUS_OK, "misplaced": misplaced}
    elif misplaced:
        results[filename] = {"status": STATUS_MISPLACED, "misplaced": misplaced}
    else:
        results[filename] = {"status": STATUS_MISSING, "misplaced": []}

print("\nFile Status Report")
print("-" * 60)

for filename, result in results.items():
    info = EXPECTED_FILES[filename]
    status = result["status"]

    if status == STATUS_OK:
        loc = info.get("correct_key", info.get("correct_prefix") + "...")
        print(f"\n[OK]       {filename}")
        print(f"           Found at correct location: {loc}")
        if result["misplaced"]:
            print(f"           NOTE: Also found at wrong location(s):")
            for mp in result["misplaced"]:
                print(f"             - {mp}")

    elif status == STATUS_MISPLACED:
        print(f"\n[MISPLACED] {filename}")
        print(f"           {info['description']}")
        print(f"           Found at WRONG location(s):")
        for mp in result["misplaced"]:
            print(f"             - {mp}")
        correct = info.get("correct_key", info.get("correct_prefix") + "<job>/output/" + filename)
        print(f"           Should be at: {correct}")
        print(f"           --> Will be fixed in the next step.")

    elif status == STATUS_MISSING:
        print(f"\n[MISSING]  {filename}")
        print(f"           {info['description']}")
        print(f"           Not found anywhere in the bucket.")
        print(f"           Created by: {info['source']}")
        print(f"           Fix: Re-run {info['fallback']}")

print("\n" + "=" * 60)

needs_fix = any(r["status"] == STATUS_MISPLACED for r in results.values())
has_missing = any(r["status"] == STATUS_MISSING for r in results.values())

if not needs_fix and not has_missing:
    print("\nAll files are in the correct location. No action needed.")
elif needs_fix:
    print("\nMisplaced files detected. Run the next cell to fix.")
if has_missing:
    print("\nMissing files detected. See instructions above for which notebooks to re-run.")

---
## Check 2: Fix misplaced files

This cell copies misplaced files to the correct `fraudshield/` location. Originals are left in place -- you can delete them manually once you confirm everything works.

**Only run this cell if the report above shows MISPLACED files.**

In [ ]:
moved_count = 0

for filename, result in results.items():
    if result["status"] != STATUS_MISPLACED:
        continue

    info = EXPECTED_FILES[filename]
    source_key = result["misplaced"][0]

    if "correct_key" in info:
        dest_key = info["correct_key"]
    else:
        dest_key = f"{info['correct_prefix']}relocated/{filename}"

    print(f"Copying: {source_key}")
    print(f"     -> {dest_key}")

    copy_s3_object(bucket, source_key, dest_key)

    try:
        s3.head_object(Bucket=bucket, Key=dest_key)
        print(f"     [OK] Verified at new location.")
        moved_count += 1
    except Exception as e:
        print(f"     [ERROR] Copy verification failed: {e}")

    print()

if moved_count == 0:
    print("No files needed to be moved.")
else:
    print(f"Moved {moved_count} file(s) to the correct location.")
    print("\nOriginals are still at their old location.")
    print("Run the verification cell below to confirm, then optionally delete the originals.")

---
## Check 3: Verify final structure

Run this after fixing to confirm everything is in place.

In [ ]:
print(f"Verifying structure under s3://{bucket}/{prefix}/\n")

checks = [
    ("train.csv",      f"{prefix}/data/train/train.csv",      "W1 Friday notebook",  "W2 Monday notebook -- Fallback cell"),
    ("validation.csv", f"{prefix}/data/validation/validation.csv", "W1 Friday notebook", "W2 Monday notebook -- Fallback cell"),
]

all_good = True

for name, key, source, fix in checks:
    try:
        s3.head_object(Bucket=bucket, Key=key)
        print(f"  [OK]      {key}")
    except Exception:
        print(f"  [MISSING] {key}")
        print(f"            Created by: {source}")
        print(f"            Fix: Re-run {fix}")
        all_good = False

model_found = False
model_key = None
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=f"{prefix}/output/"):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith("model.tar.gz") and "cnn" not in obj["Key"].lower():
            model_found = True
            model_key = obj["Key"]
            break
    if model_found:
        break

if model_found:
    print(f"  [OK]      {model_key}")
else:
    print(f"  [MISSING] {prefix}/output/<job-name>/output/model.tar.gz")
    print(f"            Random Forest model artifact")
    print(f"            Created by: W1 Friday notebook -- Launch Training Job")
    print(f"            Fix: Re-run W2 Monday notebook -- Fallback cell retrains the model")
    all_good = False

print("\n" + "=" * 60)
if all_good:
    print("\nAll checks passed. Your S3 structure is ready for the lecture notebooks.")
else:
    print("\nSome files are still missing. Follow the instructions above to fix.")

---
## Optional: Clean up misplaced originals

After verifying everything is in the correct location, run this cell to delete the misplaced copies from the bucket root. **Only run this after confirming the verification above shows all [OK].**

In [ ]:
root_keys = [k for k in list_all_keys(bucket) if not k.startswith(prefix + "/") and k != ""]

known_files = {"train.csv", "validation.csv", "model.tar.gz"}
deletable = [k for k in root_keys if any(k.endswith(f) for f in known_files)]

if not deletable:
    print("No misplaced FraudShield files found at bucket root. Nothing to delete.")
else:
    print("The following files at bucket root appear to be misplaced FraudShield files:\n")
    for k in deletable:
        print(f"  s3://{bucket}/{k}")

    print(f"\nTo delete these, uncomment and run the lines below.")
    print("(Intentionally commented out as a safety measure.)\n")

    for k in deletable:
        print(f"# s3.delete_object(Bucket=bucket, Key='{k}')")
        print(f"# print(f'Deleted: {k}')")

---

## Quick Reference: Which notebook creates what

| File | S3 Location | Created By | If Missing, Re-run |
|------|------------|------------|--------------------|
| `train.csv` | `fraudshield/data/train/train.csv` | W1 Friday notebook | W2 Monday fallback cell |
| `validation.csv` | `fraudshield/data/validation/validation.csv` | W1 Friday notebook | W2 Monday fallback cell |
| `model.tar.gz` (RF) | `fraudshield/output/<job>/output/model.tar.gz` | W1 Friday training job | W2 Monday fallback cell |
| Feature Store data | `fraudshield/feature-store/` | W2 Tuesday notebook | W2 Tuesday Steps 2-4 |
| Autopilot input | `fraudshield/autopilot/input/` | W2 Tuesday notebook | W2 Tuesday Step 6 (Autopilot) |